# AgentCore Gateway + Microsoft Graph API를 사용한 OBO 토큰 교환

이 튜토리얼에서는 **AgentCore Gateway**를 사용하여 Microsoft Graph API를 MCP 도구로 노출하고, **OBO(On-Behalf-Of) 토큰 교환**으로 인증을 투명하게 처리하는 방법을 알아봅니다. 에이전트 코드는 몇 줄이면 충분하며 토큰 처리, 세션 바인딩, Runtime 배포가 필요하지 않습니다.

### 작동 방식

1. 사용자가 **Microsoft Entra ID**로 직접 인증하고 앱 범위의 액세스 토큰(`api://<client-id>/access_as_user`)을 받습니다.
2. 사용자가 해당 Entra ID 토큰을 bearer로 **AgentCore Gateway**에 전달합니다.
3. Gateway가 Entra ID의 OIDC 검색 URL을 사용하여 토큰을 검증합니다(인바운드 인증).
4. Gateway가 **OBO 토큰 교환**을 수행하여 JWT Authorization Grant(RFC 7523)를 통해 앱 범위의 Entra ID 토큰을 Microsoft Graph 토큰으로 교환합니다.
5. **Strands 에이전트**가 Gateway MCP URL에 연결하여 도구를 검색하고 호출합니다.

**문서:** [OBO 토큰 교환](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-behalf-of-token-exchange.html) · [Gateway 아웃바운드 인증](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-outbound-auth.html)

## 사전 요구 사항

- Python 3.10+
- 구성된 AWS 자격 증명
- Amazon Bedrock AgentCore SDK
- Microsoft Entra ID에 액세스할 수 있는 Microsoft 365 **회사 또는 학교 계정**

> ⚠️ **개인 Microsoft 계정(`@outlook.com`, `@hotmail.com`, `@live.com`)으로는 캘린더/이메일 데모를 사용할 수 없습니다.** OBO 토큰 교환 자체는 개인 계정에서도 작동하지만, Microsoft Graph 캘린더 및 메일 엔드포인트에는 회사/학교 계정에서만 제공되는 Exchange Online 사서함이 필요합니다. [Microsoft 365 Developer Program](https://developer.microsoft.com/en-us/microsoft-365/dev-program)에서 무료 샌드박스를 받으세요.

In [ ]:
!pip3 install -U -r requirements.txt --quiet

---
## 1단계: Microsoft Entra ID에 애플리케이션 등록

### 1.1 앱 등록 생성
1. [entra.microsoft.com](https://entra.microsoft.com)으로 이동하여 **Identity** → **Applications** → **App registrations**를 선택합니다.
2. **+ New registration**을 클릭합니다.
3. 이름은 `AgentCore-OBO-Tutorial`, 테넌트 유형은 Single tenant로 지정합니다.
4. **Register**를 클릭합니다.

### 1.2 ID 기록
**Overview** 페이지에서 다음 값을 복사합니다.
- **Application (client) ID** → `MICROSOFT_CLIENT_ID`
- **Directory (tenant) ID** → `MICROSOFT_TENANT_ID`

### 1.3 클라이언트 보안 암호 생성
1. **Certificates & secrets** → **+ New client secret**를 선택합니다.
2. **Value**를 즉시 복사합니다 → `MICROSOFT_CLIENT_SECRET`

### 1.4 API 권한 구성
1. **API permissions** → **+ Add a permission** → **Microsoft Graph** → **Delegated permissions**를 선택합니다.
2. 다음 권한을 추가합니다: `Calendars.Read`, `Mail.Read`, `User.Read`

### 1.5 API 노출(OBO에 필요)
1. **Expose an API** → **Application ID URI**를 설정합니다(기본값 `api://<client-id>` 사용).
2. **+ Add a scope**에서 이름은 `access_as_user`, 동의 대상은 Admins and users, 상태는 Enabled로 지정합니다.

### 1.6 인증 구성(리디렉션 URI)
1. **Authentication** → **+ Add a platform** → **Web**을 선택합니다.
2. 리디렉션 URI를 추가합니다: `http://localhost:9090/oauth2/callback`
3. **Configure**를 클릭합니다.

> 이 리디렉션 URI는 토큰 콜백 서버가 권한 부여 코드를 수신하는 데 사용됩니다.

### 1.7 관리자 동의 부여
1. **API permissions** → **Grant admin consent for [tenant]** → **Yes**를 선택합니다.

> ✅ 모든 권한에 녹색 확인 표시가 나타나야 합니다.

### 중요: 토큰 버전
기본적으로 Entra ID는 **v1.0 액세스 토큰**(발급자: `https://sts.windows.net/{tenant}/`)을 발급합니다. 이 튜토리얼의 Gateway 검색 URL은 이에 맞춰 v1.0 엔드포인트를 사용합니다. v2.0 토큰이 필요한 경우 앱 매니페스트에서 `accessTokenAcceptedVersion: 2`를 설정하고 v2.0 검색 URL을 사용하세요.

---
## 2단계: OBO 자격 증명 공급자 생성

아래 `.env` 파일에 Entra ID 자격 증명을 입력한 다음 자격 증명 공급자를 생성합니다.

**중요:** `onBehalfOfTokenExchangeConfig`는 `customOauth2ProviderConfig` 내부에서만 사용할 수 있으므로, 기본 제공 `MicrosoftOAuth2` 공급자 대신 `CustomOauth2`를 공급자로 사용합니다.

In [ ]:
%%writefile .env
MICROSOFT_CLIENT_ID=""      # Entra ID의 Application (client) ID
MICROSOFT_CLIENT_SECRET=""  # Entra ID의 client secret 값
MICROSOFT_TENANT_ID=""      # Entra ID의 Directory (tenant) ID

In [ ]:
import os
import json
import time
import urllib.parse

import boto3
from boto3.session import Session
from dotenv import dotenv_values

# .env를 로드하고 환경 변수를 설정합니다.
env = dotenv_values(".env")
os.environ.update(env)

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

identity_client = boto_session.client("bedrock-agentcore-control")

In [ ]:
tenant_id = os.environ["MICROSOFT_TENANT_ID"]
client_id = os.environ["MICROSOFT_CLIENT_ID"]

# OBO 토큰 교환을 사용하는 Custom OAuth2 자격 증명 공급자를 생성합니다.
# 참고: onBehalfOfTokenExchangeConfig는 customOauth2ProviderConfig 내부에서만 사용할 수 있습니다.
microsoft_provider = identity_client.create_oauth2_credential_provider(
    name="microsoft-obo-provider-v3",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {
                "discoveryUrl": f"https://login.microsoftonline.com/{tenant_id}/v2.0/.well-known/openid-configuration"
            },
            "clientId": os.environ["MICROSOFT_CLIENT_ID"],
            "clientSecret": os.environ["MICROSOFT_CLIENT_SECRET"],
            "clientAuthenticationMethod": "CLIENT_SECRET_POST",
            "onBehalfOfTokenExchangeConfig": {"grantType": "JWT_AUTHORIZATION_GRANT"},
        }
    },
)
print("Credential provider created \u2713")
print(f"ARN: {microsoft_provider['credentialProviderArn']}")

---
## 3단계: AgentCore Gateway 생성

Gateway는 Microsoft Graph API를 MCP 도구로 변환합니다.
- **인바운드 인증**: Entra ID OIDC JWT 권한 부여자(CUSTOM_JWT)로 Entra ID에서 직접 발급한 토큰을 검증합니다.
- **대상**: 캘린더, 이메일, 프로필 엔드포인트를 MCP 도구로 노출하는 OpenAPI 스키마입니다.
- **아웃바운드 인증**: 자격 증명 공급자를 통해 OBO 토큰을 교환합니다.

**핵심 사항**: 인바운드 인증에는 Entra ID의 **v1.0** OIDC 검색 URL을 사용합니다. Entra ID는 기본적으로 v1.0 액세스 토큰(발급자: `sts.windows.net`)을 발급하므로 검색 URL도 이 버전과 일치해야 합니다. v2.0 URL을 사용하면 서명 키와 발급자가 일치하지 않아 403 오류가 발생합니다.

먼저 Gateway용 IAM 역할을 생성한 다음 Gateway 자체를 생성합니다.

In [ ]:
iam_client = boto3.client("iam")
account_id = boto3.client("sts").get_caller_identity()["Account"]

# 공유 유틸리티를 사용하여 Gateway IAM 역할을 생성합니다.
# 다른 AgentCore Gateway 튜토리얼에서 사용하는 패턴에 맞춰
# bedrock-agentcore:*, secretsmanager:GetSecretValue 등의 권한을 가진 역할을 생성합니다.
import sys

gw_utils_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "02-AgentCore-gateway"))
sys.path.insert(0, gw_utils_dir)
try:
    from utils import create_agentcore_gateway_role

    gateway_role_name = "agentcore-microsoft-obo-gateway-role"
    gateway_iam_role = create_agentcore_gateway_role("microsoft-obo-gateway")
    gateway_role_arn = gateway_iam_role["Role"]["Arn"]
    print(f"Gateway role: {gateway_role_arn}")
except ImportError:
    print("Shared utils not found, creating role inline...")
    gateway_role_name = "agentcore-obo-gateway-role"
    assume_role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:*"},
                },
            }
        ],
    }
    try:
        role_response = iam_client.create_role(
            RoleName=gateway_role_name,
            AssumeRolePolicyDocument=json.dumps(assume_role_policy),
        )
        gateway_role_arn = role_response["Role"]["Arn"]
    except iam_client.exceptions.EntityAlreadyExistsException:
        gateway_role_arn = f"arn:aws:iam::{account_id}:role/{gateway_role_name}"
    gateway_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AgentCoreAccess",
                "Effect": "Allow",
                "Action": ["bedrock-agentcore:*"],
                "Resource": "*",
            },
            {
                "Sid": "SecretsManagerAccess",
                "Effect": "Allow",
                "Action": ["secretsmanager:GetSecretValue"],
                "Resource": [f"arn:aws:secretsmanager:{region}:{account_id}:secret:*bedrock-agentcore*"],
            },
        ],
    }
    iam_client.put_role_policy(
        RoleName=gateway_role_name,
        PolicyName="GatewayOBOPolicy",
        PolicyDocument=json.dumps(gateway_policy),
    )
    print(f"Gateway role: {gateway_role_arn}")

print("Waiting for IAM propagation (10s)...")
time.sleep(10)
print("Done \u2713")

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

# 인바운드 인증용 Entra ID v1.0 OIDC 검색 URL입니다.
# 중요: Entra ID는 기본적으로 v1.0 액세스 토큰을 발급합니다(발급자: sts.windows.net).
# 검색 URL은 토큰 버전과 반드시 일치해야 하므로 v2.0이 아닌 v1.0을 사용합니다.
discovery_url = f"https://login.microsoftonline.com/{tenant_id}/.well-known/openid-configuration"

# Entra ID JWT 권한 부여자를 사용하는 Gateway를 생성합니다.
# Entra ID 액세스 토큰의 "aud" 클레임에 Application ID URI가 들어가므로
# allowedAudience에는 Application ID URI(api://<client-id>)를 사용합니다.
gateway_response = agentcore_control_client.create_gateway(
    name="microsoft-obo-gateway-v3",
    roleArn=gateway_role_arn,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedAudience": [f"api://{client_id}"],
        }
    },
    exceptionLevel="DEBUG",  # 디버깅을 위해 자세한 오류를 표시합니다.
)
gateway_id = gateway_response["gatewayId"]
print(f"Gateway created \u2713  ID: {gateway_id}")
print(f"Inbound auth: Entra ID OIDC (tenant: {tenant_id})")
print(f"Allowed audience: api://{client_id}")

### 3b단계: Gateway 대기 후 Microsoft Graph 대상 추가

대상을 추가하기 전에 Gateway가 `READY` 상태여야 합니다. 그런 다음 Microsoft Graph 엔드포인트의 OpenAPI 스키마를 정의하면 Gateway가 이를 MCP 도구로 노출합니다.

**중요 사항:**
- 매개변수 이름으로 `$top`/`$select`가 아닌 `top`과 `select`를 사용합니다. `$` 문자는 Bedrock의 도구 스키마 검증을 실패하게 합니다.
- Entra ID의 OBO 엔드포인트에는 `requested_token_use: on_behalf_of`가 포함된 `customParameters`가 **필수**입니다.

In [ ]:
# 대상을 추가하기 전에 Gateway가 READY 상태가 될 때까지 기다립니다.
end_statuses = ["READY", "FAILED", "CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"]
status = "CREATING"
print("Waiting for Gateway to become READY...")
while status not in end_statuses:
    time.sleep(10)
    gw = agentcore_control_client.get_gateway(gatewayIdentifier=gateway_id)
    status = gw["status"]
    print(f"  Gateway status: {status}")
print(f"Gateway is {status} \u2713" if status == "READY" else f"\u274c Gateway: {status}")

# Microsoft Graph 엔드포인트의 OpenAPI 사양을 정의합니다.
openapi_spec = json.dumps(
    {
        "openapi": "3.0.0",
        "info": {"title": "Microsoft Graph Calendar API", "version": "1.0"},
        "servers": [{"url": "https://graph.microsoft.com/v1.0"}],
        "paths": {
            "/me/calendarview": {
                "get": {
                    "operationId": "listCalendarEvents",
                    "summary": "List calendar events for the current user within a date range",
                    "parameters": [
                        {
                            "name": "startDateTime",
                            "in": "query",
                            "required": True,
                            "schema": {"type": "string"},
                            "description": "Start date/time in ISO 8601 format (e.g. 2025-01-01T00:00:00Z)",
                        },
                        {
                            "name": "endDateTime",
                            "in": "query",
                            "required": True,
                            "schema": {"type": "string"},
                            "description": "End date/time in ISO 8601 format (e.g. 2025-01-02T00:00:00Z)",
                        },
                    ],
                    "responses": {"200": {"description": "Calendar events"}},
                }
            },
            "/me/messages": {
                "get": {
                    "operationId": "listUserMails",
                    "summary": "List recent email messages for the current user",
                    "parameters": [
                        {
                            "name": "top",
                            "in": "query",
                            "required": False,
                            "schema": {"type": "integer"},
                            "description": "Number of messages to return",
                        },
                        {
                            "name": "select",
                            "in": "query",
                            "required": False,
                            "schema": {"type": "string"},
                            "description": "Comma-separated list of fields to return",
                        },
                    ],
                    "responses": {"200": {"description": "Email messages"}},
                }
            },
            "/me": {
                "get": {
                    "operationId": "getMyProfile",
                    "summary": "Get the current user profile information",
                    "responses": {"200": {"description": "User profile"}},
                }
            },
        },
    }
)

# OBO 아웃바운드 인증을 사용하는 Gateway 대상을 생성합니다.
target_response = agentcore_control_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="microsoft-graph-obo",
    description="Microsoft Graph API with OBO token exchange",
    targetConfiguration={"mcp": {"openApiSchema": {"inlinePayload": openapi_spec}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": microsoft_provider["credentialProviderArn"],
                    "scopes": ["https://graph.microsoft.com/.default"],
                    "grantType": "TOKEN_EXCHANGE",
                    "customParameters": {"requested_token_use": "on_behalf_of"},
                }
            },
        }
    ],
)
target_id = target_response["targetId"]
print(f"Target created \u2713  ID: {target_id}")
print(f"Target name: {target_response['name']}")

### 3c단계: 대상 대기

대상이 `READY` 상태가 될 때까지 기다립니다.

In [ ]:
# 대상이 READY 상태가 될 때까지 기다립니다.
target_end_statuses = [
    "READY",
    "FAILED",
    "UPDATE_UNSUCCESSFUL",
    "SYNCHRONIZE_UNSUCCESSFUL",
]
target_status = "CREATING"

print("Waiting for Target to become READY...")
while target_status not in target_end_statuses:
    time.sleep(10)
    tgt = agentcore_control_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    target_status = tgt["status"]
    print(f"  Target status: {target_status}")

if target_status == "READY":
    print("\nTarget is READY \u2713")
else:
    print(f"\n\u274c Target ended in status: {target_status}")

In [ ]:
# Gateway MCP URL을 가져옵니다.
gateway_details = agentcore_control_client.get_gateway(gatewayIdentifier=gateway_id)
gateway_mcp_url = gateway_details.get("gatewayUrl", "")
print(f"Gateway MCP URL: {gateway_mcp_url}")

---
## 4단계: Entra ID 액세스 토큰 가져오기

이 셀은 실행 환경을 인식하는 콜백 서버를 시작하고 브라우저에서 Entra ID 로그인 페이지를 연 다음, 로그인하면 토큰을 자동으로 캡처합니다. 복사하여 붙여 넣을 필요가 없습니다.

### 실행 환경을 인식하는 OAuth2 콜백 서버

`token_callback_server.py`는 다양한 실행 환경에 자동으로 맞춰 동작합니다.

**로컬 개발 환경:**
- 외부 콜백 URL: `http://localhost:9090/oauth2/callback`(브라우저에서 액세스 가능)
- 내부 통신: `http://localhost:9090`(Notebook ↔ 서버)
- 서버 바인딩: `127.0.0.1`(localhost 전용, 안전함)

**SageMaker Workshop Studio:**
- 외부 콜백 URL: `https://<domain>.studio.<region>.sagemaker.aws/proxy/9090/oauth2/callback`(프록시를 통해 브라우저에서 액세스 가능)
- 내부 통신: `http://localhost:9090`(동일한 컨테이너의 Notebook ↔ 서버)
- 서버 바인딩: `0.0.0.0`(SageMaker 프록시가 서버에 연결할 수 있도록 허용)

서버는 `/opt/ml/metadata/resource-metadata.json`의 존재 여부를 확인하여 환경을 감지하고 그에 맞게 자체 구성을 설정합니다.

> ⚠️ **Entra ID에 콜백 URL 등록:** Entra ID 앱 등록(1.6단계)의 리디렉션 URI는 사용 환경의 콜백 URL과 일치해야 합니다. 로컬 개발 환경에서는 `http://localhost:9090/oauth2/callback`을 사용합니다. SageMaker에서는 아래 셀이 출력하는 프록시 URL을 사용합니다.

In [ ]:
import subprocess
import webbrowser
import base64
import sys
from token_callback_server import (
    is_server_running,
    get_callback_url,
    wait_for_server_ready,
    wait_for_token,
)

bearer_token = None

# 토큰 콜백 서버가 실행 중이 아니면 시작합니다.
if is_server_running():
    print("Token callback server already running, skipping start...")
else:
    server_cmd = [
        sys.executable,
        "token_callback_server.py",
        tenant_id,
        client_id,
        os.environ["MICROSOFT_CLIENT_SECRET"],
    ]
    server_process = subprocess.Popen(server_cmd)
    if not wait_for_server_ready():
        print("\u274c Failed to start token callback server")
    else:
        print("Token callback server started \u2713")

# 실행 환경을 인식하는 콜백 URL을 사용하여 권한 부여 URL을 구성합니다.
callback_url = get_callback_url()
authorize_url = (
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/authorize?"
    f"client_id={client_id}&response_type=code&"
    f"redirect_uri={urllib.parse.quote(callback_url)}&"
    f"scope={urllib.parse.quote(f'api://{client_id}/access_as_user openid profile email')}"
)

print(f"Callback URL: {callback_url}")
print("Opening browser for Microsoft sign-in...")
webbrowser.open(authorize_url)
print("Waiting for sign-in (up to 2 minutes)...")

bearer_token = wait_for_token(timeout=120)

if bearer_token:
    payload = bearer_token.split(".")[1]
    payload += "=" * (4 - len(payload) % 4)
    claims = json.loads(base64.urlsafe_b64decode(payload))
    print(f"\n\u2705 Token captured for: {claims.get('name', 'unknown')}")
    print(f"   aud: {claims['aud']}")
    print(f"   scp: {claims.get('scp', 'N/A')}")
    print(f"   iss: {claims['iss']}")
    print(f"   ver: {claims['ver']}")
else:
    print("\u274c Timed out. Run this cell again.")

---
## 5단계: 에이전트 실행

전체 에이전트 코드는 몇 줄이면 충분합니다. `MCPClient`가 Gateway에 연결하여 MCP 도구(프로필, 캘린더, 이메일)를 검색하고, Strands Agent가 이 도구를 사용하여 사용자의 질문에 답합니다.

**`@requires_access_token`, 사용자 지정 도구, 콜백 서버, Docker가 필요하지 않습니다.**

> 💡 `getMyProfile` 도구는 모든 계정 유형에서 작동합니다. `listCalendarEvents` 및 `listUserMails` 도구를 사용하려면 Exchange Online 사서함이 있는 회사/학교 계정이 필요합니다.

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

# AgentCore Gateway에 MCP 서버로 연결합니다.
mcp_client = MCPClient(
    lambda: streamablehttp_client(gateway_mcp_url, headers={"Authorization": f"Bearer {bearer_token}"})
)

with mcp_client:
    # Gateway는 listCalendarEvents, listUserMails, getMyProfile MCP 도구를 제공합니다.
    tools = mcp_client.list_tools_sync()
    print(f"Discovered {len(tools)} MCP tools from Gateway:")
    for t in tools:
        print(f"  - {t.tool_name}")

    agent = Agent(
        model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        tools=tools,
        system_prompt="You are a Microsoft 365 assistant. Use the available tools to help users with their Microsoft profile, Outlook calendar, and email.",
    )

    response = agent("What is my Microsoft profile information?")
    print(response)

### 5b단계: 사용자 토큰과 OBO 토큰 비교

OBO 교환은 토큰을 두 가지 중요한 측면에서 변환합니다.

| | 사용자 토큰(OBO 이전) | OBO 토큰(OBO 이후) |
|---|---|---|
| **대상(`aud`)** | 사용자 앱: `api://<client-id>` | Microsoft Graph: `https://graph.microsoft.com` |
| **범위(`scp`)** | `access_as_user` | `Calendars.Read Mail.Read User.Read` |
| **신원** | 사용자의 신원 | 위임을 통해 유지되는 동일한 사용자 |

OBO 토큰은 인바운드 토큰에서 원래 사용자의 `sub`를 포함하는 `xms_st` 클레임을 통해 **위임 체인**도 전달합니다. 이를 통해 Microsoft Graph는 요청이 앱 자체가 아니라 특정 사용자를 **대신하여** 작동하는 앱에서 왔음을 알 수 있습니다.

두 토큰을 디코딩하여 차이점을 확인해 보겠습니다.

In [ ]:
import requests as req

# 결과 토큰을 확인하기 위해 OBO 교환을 수동으로 수행합니다.
obo_resp = req.post(
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
        "client_id": client_id,
        "client_secret": os.environ["MICROSOFT_CLIENT_SECRET"],
        "assertion": bearer_token,
        "scope": "https://graph.microsoft.com/Calendars.Read https://graph.microsoft.com/Mail.Read https://graph.microsoft.com/User.Read",
        "requested_token_use": "on_behalf_of",
    },
)


def decode_jwt(token):
    payload = token.split(".")[1]
    payload += "=" * (4 - len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


obo_token = obo_resp.json().get("access_token", "")
if not obo_token:
    print(f"OBO exchange failed: {obo_resp.json()}")
else:
    user_claims = decode_jwt(bearer_token)
    obo_claims = decode_jwt(obo_token)

    print(f"{'CLAIM':<20} {'USER TOKEN (app-scoped)':<40} {'OBO TOKEN (Graph-scoped)'}")
    print("=" * 100)
    for f in ["aud", "iss", "ver", "scp", "appid", "name", "email", "idp", "oid"]:
        uv = str(user_claims.get(f, "\u2014"))[:38]
        ov = str(obo_claims.get(f, "\u2014"))[:38]
        changed = " \u2190 CHANGED" if uv != ov else ""
        print(f"{f:<20} {uv:<40} {ov}{changed}")

    # 위임 클레임을 표시합니다.
    xms_st = obo_claims.get("xms_st", {})
    print("\n\u2705 OBO delegation chain:")
    print(f"   xms_st.sub (original user): {xms_st.get('sub', 'N/A')}")
    print(f"   appid (acting app):         {obo_claims.get('appid', 'N/A')}")
    print(f"   app_displayname:            {obo_claims.get('app_displayname', 'N/A')}")

---
## 리소스 정리(선택 사항)

리소스를 삭제하려면 주석을 해제하고 실행합니다.

**다음 항목도 수동으로 삭제합니다:**
- Entra 포털의 Microsoft Entra ID 앱 등록
- IAM Console의 IAM 역할(`agentcore-obo-gateway-role`)

In [ ]:
# # Gateway target 삭제
# agentcore_control_client.delete_gateway_target(
#     gatewayIdentifier=gateway_id,
#     targetId=target_id
# )
# print("Gateway target deleted \u2713")

# # Gateway 삭제
# agentcore_control_client.delete_gateway(gatewayIdentifier=gateway_id)
# print("Gateway deleted \u2713")

# # credential provider 삭제
# identity_client.delete_oauth2_credential_provider(name="microsoft-obo-provider")
# print("Credential provider deleted \u2713")

# # IAM role 삭제
# iam_client.delete_role_policy(RoleName=gateway_role_name, PolicyName="GatewayOBOPolicy")
# iam_client.delete_role(RoleName=gateway_role_name)
# print("IAM role deleted \u2713")

## 축하합니다! 🎉

**Entra ID를 직접** 사용하여 엔드 투 엔드 인증을 수행하고, OBO 토큰 교환과 함께 AgentCore Gateway를 통해 Microsoft Graph API에 액세스하는 Strands 에이전트를 구축했습니다. 에이전트 코드에는 **토큰 처리 로직이 전혀 없으며** Gateway가 모든 작업을 투명하게 처리합니다.

### 완료한 작업
- **Microsoft Entra ID**에 OBO 호환 범위가 있는 앱을 등록하고 API를 노출했습니다.
- `JWT_AUTHORIZATION_GRANT` OBO 구성이 포함된 **Custom OAuth2 자격 증명 공급자**를 생성했습니다.
- Entra ID v1.0 OIDC 인바운드 인증 및 OpenAPI 스키마 대상을 사용하는 **AgentCore Gateway**를 설정했습니다.
- `TOKEN_EXCHANGE` 권한 부여 유형과 `requested_token_use: on_behalf_of` 사용자 지정 매개변수로 대상을 구성했습니다.
- **Entra ID**로 직접 인증하여 앱 범위의 액세스 토큰을 받았습니다.
- **Strands 에이전트**를 Gateway MCP URL에 연결하고 도구를 자동으로 검색했습니다.
- 에이전트를 통해 **Microsoft 프로필**을 조회하고 Gateway가 OBO를 통해 모든 인증을 처리하도록 구성했습니다.

### 핵심 학습 내용
- Gateway의 **인바운드 인증 검색 URL은 토큰 버전과 일치해야 합니다**(Entra ID의 기본값은 v1.0).
- Entra ID v1.0 토큰용 Gateway 권한 부여자 구성에서 `allowedClients`를 사용하면 **안 됩니다**.
- Entra ID의 OBO 엔드포인트에는 `requested_token_use: on_behalf_of`가 포함된 `customParameters`가 **필수**입니다.
- IAM 역할은 다른 Gateway 튜토리얼과 동일한 `bedrock-agentcore:*` 패턴을 사용합니다(공유 `utils.create_agentcore_gateway_role` 사용).
- 개인 계정은 `/me` 프로필에서 작동하지만, 캘린더/이메일 엔드포인트에는 Exchange Online이 있는 회사/학교 계정이 필요합니다.

### 다음 단계
- **회사/학교 계정**을 사용하여 캘린더 및 이메일 도구를 테스트합니다.
- OpenAPI 스키마에 더 많은 Microsoft Graph 엔드포인트(예: OneDrive, Teams)를 추가합니다.
- 다양한 에이전트 프롬프트를 사용해 봅니다: "What meetings do I have this week?", "List my recent emails"
- [OBO 토큰 교환 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-behalf-of-token-exchange.html)와 [Gateway 아웃바운드 인증 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-outbound-auth.html)를 살펴봅니다.